# Histórico de aprendizagem
Arquivo de referência: a coleta foi executada em etapas, com repetições de células e retomadas. Não execute tudo para reproduzir a análise. Use pncp_analise_sp_final.ipynb e o banco salvo. Saídas transitórias removidas; código original preservado.

# Análise de contratações públicas — PNCP

## Objetivo
Explorar contratações públicas de órgãos do estado de São Paulo usando dados do Portal Nacional de Contratações Públicas (PNCP).

## Pergunta inicial
Como as contratações do recorte escolhido se distribuem entre municípios e modalidades de contratação?

## Recorte
Começaremos com um período curto, que será definido após consultar a documentação da API.

## Etapas previstas
Coletar dados com Python, conferir sua qualidade, armazenar em SQLite, analisar com SQL e criar gráficos com Plotly.

## Cuidados
Diferenciar valores estimados, homologados e efetivamente pagos. Registrar limitações dos dados e da análise.

In [ ]:
uf = "SP"
data_inicial = "20260801"
data_final = "20260807"

print("Estado:", uf)
print("Período:", data_inicial, "até", data_final)

In [ ]:
import requests

url = "https://pncp.gov.br/api/consulta/v1/contratacoes/publicacao"

parametros = {
    "dataInicial": data_inicial,
    "dataFinal": data_final,
    "uf": uf,
    "codigoModalidadeContratacao": 6,
    "pagina": 1,
    "tamanhoPagina": 10
}

resposta = None

try:
    resposta = requests.get(
        url,
        params=parametros,
        timeout=(10, 60)
    )
    print("Código da resposta:", resposta.status_code)

except requests.exceptions.Timeout:
    print("O PNCP não respondeu dentro do tempo limite.")

except requests.exceptions.RequestException as erro:
    print("Falha na consulta:", erro)

In [ ]:
try:
    resposta = requests.get(
        url,
        params=parametros,
        timeout=(10, 60)
    )
    print("Código da resposta:", resposta.status_code)

except requests.exceptions.Timeout:
    print("O PNCP não respondeu dentro do tempo limite.")

except requests.exceptions.RequestException as erro:
    print("Falha na consulta:", erro)

In [ ]:
dados = resposta.json()

print("Tipo recebido:", type(dados).__name__)
print("Campos disponíveis:", list(dados.keys()))

In [ ]:
registros = dados["data"]

print("Registros nesta página:", len(registros))
print("Total informado pela API:", dados["totalRegistros"])
print("Página atual:", dados["numeroPagina"])
print("Total de páginas:", dados["totalPaginas"])
print("Páginas restantes:", dados["paginasRestantes"])

In [ ]:
primeiro_registro = registros[0]

print("Campos da primeira contratação:")

for campo in primeiro_registro:
    print(campo)

In [ ]:
campos = [
    "numeroControlePNCP",
    "objetoCompra",
    "modalidadeNome",
    "dataPublicacaoPncp",
    "valorTotalEstimado",
    "valorTotalHomologado",
    "orgaoEntidade",
    "unidadeOrgao"
]

for campo in campos:
    print(campo, ":", primeiro_registro.get(campo))
    print()

In [ ]:
import pandas as pd

tabela = pd.json_normalize(registros)

print("Linhas:", tabela.shape[0])

colunas = [
    "numeroControlePNCP",
    "orgaoEntidade.razaoSocial",
    "unidadeOrgao.municipioNome",
    "unidadeOrgao.ufSigla"
]

display(tabela[colunas].head(3))

In [ ]:
campos_verificar = [
    "numeroControlePNCP",
    "unidadeOrgao.municipioNome",
    "unidadeOrgao.ufSigla",
    "valorTotalEstimado",
    "valorTotalHomologado"
]

duplicatas = tabela["numeroControlePNCP"].duplicated().sum()
ausentes = tabela[campos_verificar].isna().sum()

print("Identificadores repetidos:", duplicatas)
print("\nValores ausentes por campo:")
print(ausentes)

In [ ]:
sem_homologado = tabela["valorTotalHomologado"].isna()

conferencia = tabela.loc[
    sem_homologado,
    ["numeroControlePNCP", "situacaoCompraNome"]
]

print(conferencia.to_string(index=False))

In [ ]:
print("Registros por estado:")
print(tabela["unidadeOrgao.ufSigla"].value_counts(dropna=False))

print("\nRegistros por modalidade:")
print(tabela["modalidadeNome"].value_counts(dropna=False))

In [ ]:
datas = pd.to_datetime(
    tabela["dataPublicacaoPncp"],
    errors="coerce"
)

inicio = pd.Timestamp("2026-08-01")
fim = pd.Timestamp("2026-08-08")

fora_periodo = (datas < inicio) | (datas >= fim)

print("Data mais antiga:", datas.min())
print("Data mais recente:", datas.max())
print("Datas ausentes ou inválidas:", datas.isna().sum())
print("Registros fora do período:", fora_periodo.sum())

In [ ]:
parametros_p2 = parametros.copy()
parametros_p2["pagina"] = 2

try:
    resposta_p2 = requests.get(
        url,
        params=parametros_p2,
        timeout=(10, 60)
    )
    print("Código da resposta:", resposta_p2.status_code)

    if resposta_p2.status_code == 200:
        dados_p2 = resposta_p2.json()
        registros_p2 = dados_p2["data"]

        ids_p1 = {item["numeroControlePNCP"] for item in registros}
        ids_p2 = {item["numeroControlePNCP"] for item in registros_p2}

        print("Página retornada:", dados_p2["numeroPagina"])
        print("Registros recebidos:", len(registros_p2))
        print("IDs presentes nas duas páginas:", len(ids_p1 & ids_p2))

except requests.exceptions.RequestException as erro:
    print("Falha na consulta:", erro)

In [ ]:
registros_duas_paginas = registros + registros_p2

tabela_duas_paginas = pd.json_normalize(registros_duas_paginas)

print("Total de linhas:", len(tabela_duas_paginas))
print(
    "Identificadores repetidos:",
    tabela_duas_paginas["numeroControlePNCP"].duplicated().sum()
)

In [ ]:
paginas_extra = {}

for pagina in [3, 4]:
    filtros = parametros.copy()
    filtros["pagina"] = pagina

    try:
        retorno = requests.get(url, params=filtros, timeout=(10, 60))
        retorno.raise_for_status()

        if retorno.status_code != 200:
            print("Página sem conteúdo:", pagina)
            break

        conteudo = retorno.json()

        if conteudo["numeroPagina"] != pagina:
            print("Página diferente da solicitada. Coleta interrompida.")
            break

        paginas_extra[pagina] = conteudo["data"]
        print("Página:", pagina, "| Registros:", len(paginas_extra[pagina]))

    except requests.exceptions.RequestException as erro:
        print("Falha na página:", pagina)
        print(erro)
        break

In [ ]:
registros_quatro_paginas = registros_duas_paginas.copy()

for pagina in sorted(paginas_extra):
    registros_quatro_paginas.extend(paginas_extra[pagina])

tabela_quatro_paginas = pd.json_normalize(registros_quatro_paginas)

print("Total de linhas:", len(tabela_quatro_paginas))
print(
    "Identificadores únicos:",
    tabela_quatro_paginas["numeroControlePNCP"].nunique()
)
print(
    "Identificadores repetidos:",
    tabela_quatro_paginas["numeroControlePNCP"].duplicated().sum()
)

In [ ]:
paginas_coletadas = {
    1: registros,
    2: registros_p2,
    **paginas_extra
}

total_paginas = dados["totalPaginas"]

paginas_pendentes = [
    pagina
    for pagina in range(1, total_paginas + 1)
    if pagina not in paginas_coletadas
]

print("Páginas guardadas:", sorted(paginas_coletadas))
print("Páginas pendentes:", len(paginas_pendentes))
print("Próximas cinco:", paginas_pendentes[:5])

In [ ]:
import json
from google.colab import files

checkpoint = {
    "parametros": parametros,
    "total_paginas": total_paginas,
    "paginas_coletadas": paginas_coletadas
}

arquivo = "pncp_coleta_parcial.json"

with open(arquivo, "w", encoding="utf-8") as destino:
    json.dump(checkpoint, destino, ensure_ascii=False, indent=2)

files.download(arquivo)

In [ ]:
filtros = parametros.copy()
filtros["pagina"] = 5

try:
    retorno = requests.get(url, params=filtros, timeout=(10, 60))
    print("Código da resposta:", retorno.status_code)
    retorno.raise_for_status()

    if retorno.status_code == 200:
        conteudo = retorno.json()

        if (
            conteudo["numeroPagina"] == 5
            and conteudo["totalPaginas"] == total_paginas
        ):
            paginas_coletadas[5] = conteudo["data"]
            print("Registros recebidos:", len(conteudo["data"]))
        else:
            print("A paginação mudou. Não guardamos esta resposta.")

except requests.exceptions.RequestException as erro:
    print("A tentativa falhou:", erro)

print("Páginas na memória:", len(paginas_coletadas))

In [ ]:
import time

pendentes = [
    p for p in range(1, total_paginas + 1)
    if p not in paginas_coletadas
]

for pagina in pendentes[:20]:
    filtros = parametros.copy()
    filtros["pagina"] = pagina

    try:
        retorno = requests.get(url, params=filtros, timeout=(10, 60))
        retorno.raise_for_status()

        if retorno.status_code != 200:
            print("Resposta sem conteúdo. Paramos na página:", pagina)
            break

        conteudo = retorno.json()

        if conteudo["numeroPagina"] != pagina:
            print("Página inesperada. Coleta interrompida.")
            break

        if conteudo["totalPaginas"] != total_paginas:
            print("O total de páginas mudou. Vamos conferir antes de continuar.")
            break

        paginas_coletadas[pagina] = conteudo["data"]
        print("Página:", pagina, "| Registros:", len(conteudo["data"]))
        time.sleep(1)

    except requests.exceptions.RequestException as erro:
        print("Falha na página:", pagina, "|", erro)
        break

print("Páginas na memória:", len(paginas_coletadas))

In [ ]:
import json
from google.colab import files

checkpoint = {
    "parametros": parametros.copy(),
    "total_paginas": total_paginas,
    "paginas_coletadas": paginas_coletadas
}

arquivo = f"pncp_coleta_parcial_{len(paginas_coletadas)}_paginas.json"

with open(arquivo, "w", encoding="utf-8") as destino:
    json.dump(checkpoint, destino, ensure_ascii=False, indent=2)

files.download(arquivo)

In [ ]:
faltantes = [
    p for p in range(1, total_paginas + 1)
    if p not in paginas_coletadas
]

registros_completos = []

for pagina in sorted(paginas_coletadas):
    registros_completos.extend(paginas_coletadas[pagina])

tabela_completa = pd.json_normalize(registros_completos)

print("Páginas faltantes:", faltantes)
print("Registros reunidos:", len(tabela_completa))
print("Total informado inicialmente:", dados["totalRegistros"])
print(
    "Identificadores ausentes:",
    tabela_completa["numeroControlePNCP"].isna().sum()
)
print(
    "Identificadores repetidos:",
    tabela_completa["numeroControlePNCP"].duplicated().sum()
)

In [ ]:
datas = pd.to_datetime(
    tabela_completa["dataPublicacaoPncp"],
    errors="coerce"
)

fora_periodo = (
    (datas < pd.Timestamp("2026-08-01"))
    | (datas >= pd.Timestamp("2026-08-08"))
)

print("Registros por estado:")
print(tabela_completa["unidadeOrgao.ufSigla"].value_counts(dropna=False))

print("\nRegistros por modalidade:")
print(tabela_completa["modalidadeNome"].value_counts(dropna=False))

print("\nDatas ausentes ou inválidas:", datas.isna().sum())
print("Registros fora do período:", fora_periodo.sum())
print("Data mais antiga:", datas.min())
print("Data mais recente:", datas.max())

In [ ]:
campos_valores = [
    "valorTotalEstimado",
    "valorTotalHomologado"
]

for campo in campos_valores:
    original = tabela_completa[campo]
    numerico = pd.to_numeric(original, errors="coerce")

    print("\nCampo:", campo)
    print("Ausentes:", original.isna().sum())
    print("Preenchidos não convertíveis:", (
        original.notna() & numerico.isna()
    ).sum())
    print("Valores iguais a zero:", numerico.eq(0).sum())
    print("Valores negativos:", numerico.lt(0).sum())

In [ ]:
estimados = pd.to_numeric(
    tabela_completa["valorTotalEstimado"],
    errors="coerce"
)

estimado_zero = estimados.eq(0)

print("Situações dos registros com valor estimado zero:")
print(
    tabela_completa.loc[
        estimado_zero, "situacaoCompraNome"
    ].value_counts(dropna=False)
)

In [ ]:
campos = ["valorTotalEstimado", "valorTotalHomologado"]

valores = tabela_completa[campos].apply(
    pd.to_numeric, errors="coerce"
)

resumo_qualidade = pd.DataFrame({
    "ausentes": valores.isna().sum(),
    "zeros": valores.eq(0).sum(),
    "positivos": valores.gt(0).sum(),
    "negativos": valores.lt(0).sum()
})

display(resumo_qualidade)

In [ ]:
mapa_colunas = {
    "numeroControlePNCP": "id_contratacao",
    "orgaoEntidade.razaoSocial": "orgao",
    "unidadeOrgao.municipioNome": "municipio",
    "unidadeOrgao.ufSigla": "uf",
    "objetoCompra": "objeto",
    "modalidadeNome": "modalidade",
    "situacaoCompraNome": "situacao",
    "dataPublicacaoPncp": "data_publicacao",
    "valorTotalEstimado": "valor_estimado",
    "valorTotalHomologado": "valor_homologado"
}

tabela_sql = tabela_completa[list(mapa_colunas)].rename(
    columns=mapa_colunas
).copy()

print("Linhas:", len(tabela_sql))
print("Colunas:", tabela_sql.columns.tolist())

In [ ]:
print("Quantidade de colunas:", len(tabela_sql.columns))
print("Tem valor_estimado:", "valor_estimado" in tabela_sql.columns)
print("Tem valor_homologado:", "valor_homologado" in tabela_sql.columns)

In [ ]:
import sqlite3

conexao = sqlite3.connect("pncp_sp.sqlite")

tabela_sql.to_sql(
    "contratacoes",
    conexao,
    if_exists="replace",
    index=False
)

consulta = """
SELECT COUNT(*) AS total_registros
FROM contratacoes;
"""

display(pd.read_sql_query(consulta, conexao))

In [ ]:
from google.colab import files

conexao.commit()
files.download("pncp_sp.sqlite")

In [ ]:
consulta = """
SELECT
    COUNT(*) AS total_registros,
    COUNT(valor_homologado) AS homologados_preenchidos,
    COUNT(*) - COUNT(valor_homologado) AS homologados_ausentes
FROM contratacoes;
"""

display(pd.read_sql_query(consulta, conexao))

In [ ]:
consulta_municipios = """
SELECT
    municipio,
    COUNT(*) AS quantidade
FROM contratacoes
GROUP BY municipio
ORDER BY quantidade DESC, municipio ASC
LIMIT 10;
"""

ranking_municipios = pd.read_sql_query(
    consulta_municipios, conexao
)

display(ranking_municipios)

In [ ]:
import plotly.express as px

fig = px.bar(
    ranking_municipios,
    x="quantidade",
    y="municipio",
    orientation="h",
    text="quantidade",
    title="Pregões eletrônicos: 10 municípios com mais registros",
    labels={
        "quantidade": "Quantidade de registros",
        "municipio": "Município da unidade"
    }
)

fig.update_yaxes(
    categoryorder="array",
    categoryarray=ranking_municipios["municipio"].tolist(),
    autorange="reversed"
)

fig.update_layout(
    height=550,
    margin=dict(l=10, r=20, t=80, b=50)
)
fig.update_layout(
    title=dict(
        text="Top 10 municípios<br>Pregões eletrônicos — SP<br>Publicações: 01 a 07/08/2026",
        font=dict(size=14),
        x=0
    ),
    xaxis_title="Registros",
    yaxis_title=None,
    font=dict(size=11),
    margin=dict(l=10, r=35, t=100, b=50)
)

fig.update_xaxes(tickfont=dict(size=10))
fig.show()

In [ ]:
from google.colab import files

arquivos_enviados = files.upload()

In [ ]:
import sqlite3
import pandas as pd

conexao = sqlite3.connect(
    "file:pncp_sp.sqlite?mode=ro",
    uri=True
)

consulta = """
SELECT COUNT(*) AS total_registros
FROM contratacoes;
"""

display(pd.read_sql_query(consulta, conexao))

In [ ]:
consulta_participacao = """
SELECT
    municipio,
    COUNT(*) AS quantidade,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM contratacoes),
        2
    ) AS percentual_total
FROM contratacoes
GROUP BY municipio
ORDER BY quantidade DESC, municipio ASC
LIMIT 10;
"""

participacao_municipios = pd.read_sql_query(
    consulta_participacao, conexao
)

display(participacao_municipios)

## Primeira conclusão: distribuição por município

No recorte de pregões eletrônicos publicados de 01 a 07/08/2026,
com unidades de órgãos localizadas em SP, foram coletados
1.660 registros, sem identificadores repetidos.

São Paulo apresentou a maior quantidade: 374 registros,
equivalentes a 22,53% do total. Campinas teve 42 registros
(2,53%) e Bauru, 38 (2,29%).

A localização corresponde ao município da unidade do órgão.
Os registros podem envolver órgãos municipais, estaduais
e federais, não apenas prefeituras.

Esse resultado mede quantidade de registros, não valores
gastos, e não permite concluir o motivo da concentração.

In [ ]:
from google.colab import files

arquivos_enviados = files.upload()

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

nome_arquivo = next(iter(arquivos_enviados))
caminho = Path(nome_arquivo).resolve()

conexao = sqlite3.connect(
    caminho.as_uri() + "?mode=ro",
    uri=True
)

display(pd.read_sql_query(
    "SELECT COUNT(*) AS total_registros FROM contratacoes;",
    conexao
))

In [ ]:
consulta_situacoes = """
SELECT
    situacao,
    COUNT(*) AS quantidade,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM contratacoes),
        2
    ) AS percentual
FROM contratacoes
GROUP BY situacao
ORDER BY quantidade DESC, situacao ASC;
"""

resumo_situacoes = pd.read_sql_query(
    consulta_situacoes, conexao
)

display(resumo_situacoes)

In [ ]:
consulta = """
SELECT
    situacao,
    COUNT(*) AS total,
    COUNT(valor_homologado) AS com_valor,
    COUNT(*) - COUNT(valor_homologado) AS sem_valor
FROM contratacoes
GROUP BY situacao
ORDER BY total DESC;
"""

display(pd.read_sql_query(consulta, conexao))

In [ ]:
consulta_ausencia = """
SELECT
    situacao,
    COUNT(*) AS total,
    COUNT(*) - COUNT(valor_homologado) AS sem_valor,
    ROUND(
        100.0 * (COUNT(*) - COUNT(valor_homologado))
        / COUNT(*),
        2
    ) AS percentual_ausente
FROM contratacoes
GROUP BY situacao
ORDER BY percentual_ausente DESC, situacao ASC;
"""

ausencia_por_situacao = pd.read_sql_query(
    consulta_ausencia, conexao
)

display(ausencia_por_situacao)

## Segunda conclusão: situações e valores ausentes

Dos 1.660 registros, 1.588 (95,66%) estavam na situação
“Divulgada no PNCP” no momento da coleta.

O percentual de ausência de valor homologado dentro de
cada situação foi:

- Anulada: 100% — 10 de 10 registros.
- Suspensa: 95,45% — 42 de 44 registros.
- Revogada: 94,44% — 17 de 18 registros.
- Divulgada no PNCP: 54,97% — 873 de 1.588 registros.

Essas proporções descrevem o preenchimento dos dados.
Não demonstram que a situação causou a ausência.

Os grupos têm tamanhos diferentes. “Divulgada no PNCP”
tem a menor proporção de ausência, mas a maior quantidade
absoluta de valores ausentes: 873.

Valores ausentes foram preservados, sem substituição
por zero. Valor homologado não representa valor pago.

In [ ]:
consulta_publicacoes = """
SELECT
    SUBSTR(data_publicacao, 1, 10) AS dia,
    COUNT(*) AS quantidade
FROM contratacoes
GROUP BY SUBSTR(data_publicacao, 1, 10)
ORDER BY dia;
"""

publicacoes_por_dia = pd.read_sql_query(
    consulta_publicacoes, conexao
)

display(publicacoes_por_dia)
print("Total:", publicacoes_por_dia["quantidade"].sum())

In [ ]:
calendario = pd.DataFrame({
    "dia": pd.date_range("2026-08-01", "2026-08-07")
        .strftime("%Y-%m-%d")
})

publicacoes_semana = calendario.merge(
    publicacoes_por_dia,
    on="dia",
    how="left"
)

publicacoes_semana["quantidade"] = (
    publicacoes_semana["quantidade"].fillna(0).astype(int)
)

display(publicacoes_semana)

In [ ]:
print("Quantidade de dias:", len(publicacoes_semana))
print("Total de registros:", publicacoes_semana["quantidade"].sum())
print("\nÚltimo dia:")
print(publicacoes_semana.tail(1).to_string(index=False))

In [ ]:
import plotly.express as px

fig_diario = px.bar(
    publicacoes_semana,
    x="dia",
    y="quantidade",
    text="quantidade",
    labels={"dia": "Publicação", "quantidade": "Registros"}
)

fig_diario.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig_diario.update_layout(
    title=dict(
        text="Publicações por dia<br>Pregões eletrônicos — SP<br>01 a 07/08/2026",
        font=dict(size=14)
    ),
    height=450,
    margin=dict(l=15, r=15, t=100, b=60),
    yaxis_range=[0, 420]
)

fig_diario.update_xaxes(
    type="category",
    tickvals=publicacoes_semana["dia"].tolist(),
    ticktext=["01/08", "02/08", "03/08", "04/08",
              "05/08", "06/08", "07/08"]
)

fig_diario.show()

Publicações por dia

No recorte de pregões eletrônicos de SP, entre 01 e 07/08/2026, foram identificados 1.660 registros. O maior volume ocorreu em 05/08, com 360 registros.

Em 01/08, não foram encontrados registros no recorte coletado; em 02/08, foi encontrado apenas um. O calendário completo foi mantido no gráfico para mostrar também o dia sem registros.

Esta análise representa a quantidade de publicações, não os valores gastos ou pagos. O período de uma semana não permite afirmar uma tendência de longo prazo nem explicar as causas das diferenças entre os dias.